# Índice de incertidumbre de política comercial **sobre México** con GDELT

Este notebook construye desde cero un índice al estilo de Caldara, Iacoviello et al. (TPU), con un filtro adicional: **el artículo debe mencionar a México**. Usa la API DOC 2.0 de GDELT, que indexa el texto completo de noticias en línea de todo el mundo desde enero de 2017 (los artículos en otros idiomas se buscan en su traducción automática al inglés).

**Se ejecuta en su computadora** (requiere internet). La descarga completa tarda entre 1.5 y 3 horas por el límite de GDELT (una solicitud cada 5 segundos). Cada respuesta se guarda en disco: si se interrumpe, vuelva a ejecutar y continúa donde se quedó.

## Lógica

**1. Qué se cuenta.** Para cada mes, el número de artículos que cumplen **las tres condiciones a la vez**:

| Grupo | Caldara et al. (ProQuest) | Traducción a GDELT |
|---|---|---|
| Tema comercial | "tariff\*", "import duty", "import barrier", "trade treaty", "trade policy", "trade act", dumping, "import fee", tax n/10 ("foreign goods" OR "foreign oil" OR import\*), (imports OR import OR imported) n/10 (surtax OR surcharge\*) | GDELT no tiene comodín `*`: se listan las variantes (tariff, tariffs; "import duties"…). La cercanía `n/10` se traduce al operador de GDELT `near10:"palabra1 palabra2"`. |
| Incertidumbre | uncertainty, uncertain, risk\*, dubious, unclear, potential, probabl\*, predict\*, danger\* | Se listan las variantes de cada raíz (risk, risks, risky, risked…). «unpredictable» **no** se incluye: no empieza con «predict». |
| **México (filtro nuevo)** | — | Mexico, Mexican, Mexicans (GDELT busca palabras completas: «Mexico» no encuentra «Mexican»). |

**2. De qué medios.** Se construyen tres versiones, porque GDELT permite filtrar por país del medio (`sourcecountry`):

- **EE.UU.:** medios estadounidenses. Es la más cercana al TPU original, que usa prensa de EE.UU. **Recomendada como principal.**
- **México:** medios mexicanos (en español, buscados en su traducción). Casi todos sus artículos mencionan a México, así que ahí el filtro casi no actúa.
- **Global:** todos los medios que monitorea GDELT.

**3. Contra qué se normaliza.** Igual que Caldara (artículos que cumplen / total de artículos × 10,000; 100 = 1% de los artículos), con dos denominadores:

- **Total de artículos de los medios del mismo país** (consulta solo con `sourcecountry`). Es el equivalente directo de `N7`.
- **Total de artículos monitoreados por GDELT** (campo `norm` de la API). Se usa en la versión global.

GDELT ha ido agregando medios con el tiempo; normalizar por el total corrige ese crecimiento.

**4. Validación.** Se construye además la versión **sin** el filtro de México con medios de EE.UU. (el equivalente GDELT del TPU) y se compara con el TPU de Caldara et al. que ya tenemos. Si el método funciona, deben moverse juntos.

**5. Diferencias que no se pueden evitar.** Son noticias en línea, no los 7 periódicos impresos; la cobertura empieza en 2017; GDELT busca en la traducción automática de textos no ingleses, y cuenta notas repetidas por sindicación. Por eso el índice es **una aproximación al método**, no una réplica exacta.

## 0. Configuración
Ajuste solo este bloque si hace falta. Ejecute el notebook desde la carpeta `code/Python/` del repositorio (o desde cualquier subcarpeta: la raíz se busca automáticamente).

In [ ]:
import json, time, re, calendar, hashlib, platform
from datetime import datetime, date
from pathlib import Path

import numpy as np
import pandas as pd
import requests

INICIO = "2017-01"          # primer mes del archivo de GDELT DOC 2.0
FIN = None                  # None = último mes completo
PAUSA_SEG = 6.0             # GDELT pide una solicitud cada 5 segundos como máximo
MAX_REINTENTOS = 8
TIMEOUT_SEG = 120
URL = "https://api.gdeltproject.org/api/v2/doc/doc"
USAR_CERCANIA = True        # cláusulas near10: (se desactivan solas si GDELT rechaza la consulta por larga)

def _raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "data").is_dir() and (q / "docs").is_dir():
            return q
    return Path.cwd().resolve()
RAIZ = _raiz()
CACHE = RAIZ / "data/raw/incertidumbre/gdelt_cache"
SALIDA = RAIZ / "data/interim/incertidumbre"
CACHE.mkdir(parents=True, exist_ok=True); SALIDA.mkdir(parents=True, exist_ok=True)
hoy = date.today()
if FIN is None:
    FIN = (pd.Timestamp(hoy.year, hoy.month, 1) - pd.offsets.MonthBegin(1)).strftime("%Y-%m")
MESES = pd.period_range(INICIO, FIN, freq="M")
print(f"Raíz: {RAIZ}\nCaché: {CACHE}\nMeses: {MESES[0]} a {MESES[-1]} ({len(MESES)})")

## 1. Términos y consultas
Las listas reproducen los términos de Caldara et al. Cada cambio de traducción queda a la vista en la tabla.

In [ ]:
COMERCIO = ['tariff', 'tariffs', '"import duty"', '"import duties"', '"import barrier"', '"import barriers"',
            '"trade treaty"', '"trade treaties"', '"trade policy"', '"trade policies"', '"trade act"',
            'dumping', '"import fee"', '"import fees"']
# tax n/10 ("foreign goods" OR "foreign oil" OR import*)
CERCANIA = [f'near10:"tax {w}"' for w in ["foreign goods", "foreign oil", "import", "imports", "imported", "importing", "importer", "importers"]]
# (imports OR import OR imported) n/10 (surtax OR surcharge*)
CERCANIA += [f'near10:"{a} {b}"' for a in ["import", "imports", "imported"] for b in ["surtax", "surcharge", "surcharges"]]
INCERTIDUMBRE = ['uncertainty', 'uncertainties', 'uncertain', 'risk', 'risks', 'risky', 'risked', 'riskier', 'riskiest', 'risking',
                 'dubious', 'unclear', 'potential', 'potentially', 'probable', 'probably', 'probability', 'probabilities',
                 'predict', 'predicts', 'predicted', 'predicting', 'prediction', 'predictions', 'predictable', 'predictive',
                 'danger', 'dangers', 'dangerous', 'dangerously']
MEXICO = ['Mexico', 'Mexican', 'Mexicans']

def grupo(terminos):
    return "(" + " OR ".join(terminos) + ")"

def consulta(con_mexico=True, pais=None, cercania=None):
    cercania = USAR_CERCANIA if cercania is None else cercania
    partes = [grupo(COMERCIO + (CERCANIA if cercania else [])), grupo(INCERTIDUMBRE)]
    if con_mexico:
        partes.append(grupo(MEXICO))
    if pais:
        partes.append(f"sourcecountry:{pais}")
    return " ".join(partes)

pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame([
    ("Tema comercial", 'tariff*', "tariff, tariffs"),
    ("Tema comercial", '"import duty", "import barrier", "trade treaty", "trade policy", "import fee"', "frase en singular y plural"),
    ("Tema comercial", 'tax n/10 ("foreign goods" OR "foreign oil" OR import*)', 'near10:"tax foreign goods", near10:"tax import"… (8 cláusulas)'),
    ("Tema comercial", "(imports OR import OR imported) n/10 (surtax OR surcharge*)", 'near10:"import surtax"… (9 cláusulas)'),
    ("Incertidumbre", "risk*, probabl*, predict*, danger*", "variantes listadas (sin «unpredictable»)"),
    ("México", "(nuevo)", "Mexico OR Mexican OR Mexicans"),
], columns=["grupo", "Caldara et al.", "GDELT"]))
print("Consulta principal (medios de EE.UU.):\n", consulta(True, "US"), "\nLongitud:", len(consulta(True, "US")), "caracteres")

## 2. Series que se descargan
Cada serie es una consulta que se hace **mes por mes** (una solicitud por mes; GDELT devuelve conteos diarios que luego se suman). Los denominadores por país (consultas solo con `sourcecountry`) se agregan en la sección 4 **si el diagnóstico confirma que GDELT los acepta**.

In [ ]:
def construir_series():
    return {
        "tpu_mx_eeuu":   dict(q=consulta(True, "US"),  desc="TPU + México, medios de EE.UU. (principal)"),
        "tpu_mx_mexico": dict(q=consulta(True, "MX"),  desc="TPU + México, medios mexicanos"),
        "tpu_mx_global": dict(q=consulta(True, None),  desc="TPU + México, todos los medios"),
        "tpu_eeuu":      dict(q=consulta(False, "US"), desc="TPU sin filtro de México, medios de EE.UU. (validación contra Caldara)"),
    }
SERIES = construir_series()
DENOMINADORES = {
    "total_eeuu":    dict(q="sourcecountry:US", desc="Denominador: todos los artículos de medios de EE.UU."),
    "total_mexico":  dict(q="sourcecountry:MX", desc="Denominador: todos los artículos de medios mexicanos"),
}
display(pd.DataFrame([(k, v["desc"], len(v["q"])) for k, v in SERIES.items()], columns=["serie", "descripción", "longitud de la consulta"]))
n_sol = (len(SERIES) + len(DENOMINADORES)) * len(MESES)
print(f"Solicitudes necesarias (con denominadores por país): {n_sol} (≈ {n_sol * PAUSA_SEG / 3600:.1f} horas sin contar reintentos)")

## 3. Cliente de la API con caché, pausa y reintentos
- Cada respuesta se guarda en `data/raw/incertidumbre/gdelt_cache/<serie>/<AAAAMM>.json` junto con la consulta exacta y la fecha de descarga. Si el archivo existe y la consulta no cambió, no se vuelve a pedir.
- Se espera `PAUSA_SEG` entre solicitudes. Ante un 429 (demasiadas solicitudes) o un error del servidor, se espera cada vez más (30 s, 60 s, …).
- GDELT responde con **texto plano** cuando la consulta tiene un problema (por ejemplo, una frase demasiado corta o común). En ese caso el proceso se detiene y muestra el mensaje, en lugar de guardar un resultado vacío.

In [ ]:
_ultima = [0.0]
def _esperar():
    falta = PAUSA_SEG - (time.time() - _ultima[0])
    if falta > 0:
        time.sleep(falta)
    _ultima[0] = time.time()

def pedir(query, inicio, fin, mode="timelinevolraw", extra=None):
    params = {"query": query, "mode": mode, "format": "json",
              "startdatetime": inicio.strftime("%Y%m%d%H%M%S"), "enddatetime": fin.strftime("%Y%m%d%H%M%S")}
    if extra:
        params.update(extra)
    for k in range(MAX_REINTENTOS):
        _esperar()
        try:
            r = requests.get(URL, params=params, timeout=TIMEOUT_SEG)
        except requests.RequestException as e:
            espera = 30 * (k + 1); print(f"  error de red ({e.__class__.__name__}); reintento en {espera} s"); time.sleep(espera); continue
        if r.status_code == 429 or r.status_code >= 500:
            espera = 30 * (k + 1); print(f"  HTTP {r.status_code}; reintento en {espera} s"); time.sleep(espera); continue
        texto = r.text.strip()
        if not texto:
            return {}                       # sin resultados
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            raise RuntimeError(f"GDELT rechazó la consulta (HTTP {r.status_code}): {texto[:500]}")
    raise RuntimeError("Se agotaron los reintentos. Espere unos minutos y vuelva a ejecutar: la caché conserva lo ya descargado.")

def huella(query):
    return hashlib.sha1(query.encode()).hexdigest()[:12]

def descargar_mes(serie, query, periodo):
    archivo = CACHE / serie / f"{periodo.strftime('%Y%m')}.json"
    if archivo.exists():
        guardado = json.loads(archivo.read_text(encoding="utf-8"))
        if guardado.get("huella_consulta") == huella(query):
            return guardado["respuesta"]
    inicio = periodo.start_time.to_pydatetime()
    fin = periodo.end_time.floor("s").to_pydatetime()
    resp = pedir(query, inicio, fin)
    archivo.parent.mkdir(parents=True, exist_ok=True)
    archivo.write_text(json.dumps({"serie": serie, "consulta": query, "huella_consulta": huella(query), "mes": str(periodo),
                                   "descargado": datetime.now().isoformat(timespec="seconds"), "respuesta": resp}, ensure_ascii=False), encoding="utf-8")
    return resp

def a_diario(resp):
    """Convierte la respuesta de timelinevolraw en filas diarias (fecha, value, norm)."""
    if not resp or "timeline" not in resp or not resp["timeline"]:
        return pd.DataFrame(columns=["fecha", "value", "norm"])
    datos = resp["timeline"][0].get("data", [])
    df = pd.DataFrame(datos)
    if df.empty:
        return pd.DataFrame(columns=["fecha", "value", "norm"])
    df["fecha"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d")
    return df[["fecha", "value", "norm"]]

## 4. Diagnóstico antes de descargar todo
Se prueba cada consulta en **mayo de 2019** (amenaza de aranceles de EE.UU. a México, 30 de mayo), comprobando que:

1. GDELT acepta la consulta (sintaxis, longitud, frases).
2. La respuesta es diaria y trae `value` y `norm`.
3. Las cláusulas de cercanía aportan algo (se compara con la versión sin ellas).
4. Los artículos encontrados tienen sentido: se listan 15 títulos de la serie principal.

Si algo falla aquí, **no siga**: revise el mensaje.

In [ ]:
mayo = pd.Period("2019-05", freq="M")
ini_m, fin_m = mayo.start_time.to_pydatetime(), mayo.end_time.floor("s").to_pydatetime()
def diagnosticar(series):
    filas, resp_d = [], {}
    for serie, s_ in series.items():
        try:
            resp = pedir(s_["q"], ini_m, fin_m)
            error = ""
        except RuntimeError as e:
            resp, error = {}, str(e)[:200]
        resp_d[serie] = d = a_diario(resp)
        filas.append(dict(serie=serie, aceptada=error == "", error=error,
                          resolucion=resp.get("query_details", {}).get("date_resolution", "?") if resp else "-",
                          dias=len(d), articulos=int(d.value.sum()) if len(d) else 0, norm_total_mes=int(d.norm.sum()) if len(d) else 0,
                          articulos_30_mayo=int(d.loc[d.fecha == "2019-05-30", "value"].sum()) if len(d) else 0,
                          articulos_31_mayo=int(d.loc[d.fecha == "2019-05-31", "value"].sum()) if len(d) else 0))
    return pd.DataFrame(filas), resp_d

diag, respuestas = diagnosticar({**SERIES, **DENOMINADORES})
if not diag[diag.serie.isin(SERIES)].aceptada.all() and USAR_CERCANIA:
    print("GDELT rechazó la consulta completa; se repite sin las cláusulas de cercanía (near10:).")
    USAR_CERCANIA = False
    SERIES = construir_series()
    d2, r2 = diagnosticar(SERIES)
    diag = pd.concat([d2, diag[diag.serie.isin(DENOMINADORES)]], ignore_index=True)
    respuestas.update(r2)
print("Cláusulas de cercanía activas:", USAR_CERCANIA)
display(diag)
principales = diag[diag.serie.isin(SERIES)]
assert principales.aceptada.all(), "GDELT rechazó alguna consulta principal: lea la columna 'error'."
assert (principales.resolucion == "day").all() and (principales.dias >= 28).all(), "Alguna consulta principal no devolvió datos diarios completos."

# ¿Qué mide 'norm'? Si es igual con y sin filtro de país, es el total global de GDELT.
n_global = respuestas["tpu_mx_global"].set_index("fecha").norm
n_eeuu = respuestas["tpu_mx_eeuu"].set_index("fecha").norm
NORM_ES_GLOBAL = bool(np.allclose(n_global.reindex(n_eeuu.index), n_eeuu))
DENOM_PAIS_OK = bool(diag.loc[diag.serie.isin(DENOMINADORES), "aceptada"].all() and (diag.loc[diag.serie.isin(DENOMINADORES), "articulos"] > 0).all())
if DENOM_PAIS_OK:
    SERIES.update(DENOMINADORES)
    DECISION_DENOM = "Denominador por país: consultas solo con sourcecountry (aceptadas por GDELT)."
elif not NORM_ES_GLOBAL:
    DECISION_DENOM = "GDELT no aceptó las consultas solo con sourcecountry, pero 'norm' cambia con el filtro de país: se usa 'norm' de cada serie como total de su país."
else:
    DECISION_DENOM = ("GDELT no aceptó las consultas solo con sourcecountry y 'norm' es el total global: los índices por país se normalizan "
                      "con el total global (REVISAR: no es el equivalente exacto de N7).")
print(f"'norm' es el total global de GDELT: {NORM_ES_GLOBAL}\nDenominadores por país disponibles: {DENOM_PAIS_OK}\nDecisión: {DECISION_DENOM}")

if USAR_CERCANIA:
    sin_cerc = a_diario(pedir(consulta(True, "US", cercania=False), ini_m, fin_m))
    con_cerc = int(respuestas["tpu_mx_eeuu"].value.sum())
    print(f"Serie principal, mayo 2019: {con_cerc} artículos con cláusulas de cercanía; {int(sin_cerc.value.sum())} sin ellas (aportan {con_cerc - int(sin_cerc.value.sum())}).")

try:
    art = pedir(SERIES["tpu_mx_eeuu"]["q"], pd.Timestamp("2019-05-29").to_pydatetime(), pd.Timestamp("2019-06-02 23:59:59").to_pydatetime(),
                mode="artlist", extra={"maxrecords": 15, "sort": "hybridrel"})
    display(pd.DataFrame(art["articles"])[["seendate", "domain", "title"]] if art.get("articles") else "GDELT no devolvió lista de artículos para este periodo.")
except RuntimeError as e:
    print("No se pudo obtener la lista de artículos de ejemplo:", e)

## 5. Descarga completa (mes por mes, reanudable)

In [ ]:
t0 = time.time()
for serie, s in SERIES.items():
    pendientes = [p for p in MESES if not (CACHE / serie / f"{p.strftime('%Y%m')}.json").exists()]
    print(f"{serie}: {len(MESES) - len(pendientes)} meses ya en caché, {len(pendientes)} por descargar")
    for i, p in enumerate(MESES, 1):
        descargar_mes(serie, s["q"], p)
        if i % 12 == 0:
            print(f"  {serie}: {p} ({(time.time() - t0) / 60:.0f} min)")
print(f"Descarga terminada en {(time.time() - t0) / 60:.1f} minutos.")

## 6. Series mensuales y comprobaciones
Se suman los conteos diarios por mes. Se marca la **completitud**: días del mes con datos (`norm` > 0). Un mes con menos de 90% de días con datos se marca y **no se usa** en el índice (queda como faltante; no se imputa).

In [ ]:
filas = []
for serie, s in SERIES.items():
    for p in MESES:
        d = a_diario(descargar_mes(serie, s["q"], p))
        filas.append(dict(mes=p.to_timestamp(), serie=serie, articulos=d.value.sum() if len(d) else 0,
                          total_gdelt=d.norm.sum() if len(d) else np.nan,
                          dias_con_datos=int((d.norm > 0).sum()) if len(d) else 0, dias_mes=p.days_in_month))
largo = pd.DataFrame(filas)
ancho = largo.pivot(index="mes", columns="serie", values="articulos")
norm_serie = largo.pivot(index="mes", columns="serie", values="total_gdelt")
# Completitud: días con datos según el total monitoreado de la serie global (no depende de que haya artículos encontrados)
ref = largo[largo.serie == "tpu_mx_global"].set_index("mes")
comp = pd.DataFrame({"total_gdelt": ref.total_gdelt, "dias_con_datos": ref.dias_con_datos, "dias_mes": ref.dias_mes})
comp["completitud"] = comp.dias_con_datos / comp.dias_mes
if not DENOM_PAIS_OK:
    if not NORM_ES_GLOBAL:
        ancho["total_eeuu"], ancho["total_mexico"] = norm_serie["tpu_mx_eeuu"], norm_serie["tpu_mx_mexico"]
    else:
        ancho["total_eeuu"] = ancho["total_mexico"] = comp.total_gdelt
mensual = ancho.join(comp)
mensual["mes_completo"] = mensual.completitud >= 0.9

checks = pd.DataFrame([
    dict(prueba="Meses descargados", resultado=f"{len(mensual)} ({mensual.index.min():%Y-%m} a {mensual.index.max():%Y-%m})", estado="OK"),
    dict(prueba="Denominador por país", resultado=DECISION_DENOM, estado="OK" if DENOM_PAIS_OK else "REVISAR"),
    dict(prueba="Meses con al menos 90% de días con datos", resultado=f"{int(mensual.mes_completo.sum())} de {len(mensual)}",
         estado="OK" if mensual.mes_completo.all() else "REVISAR"),
    dict(prueba="Denominadores por país positivos en meses completos",
         resultado=f"EE.UU.: {int((mensual.loc[mensual.mes_completo, 'total_eeuu'] > 0).sum())}; México: {int((mensual.loc[mensual.mes_completo, 'total_mexico'] > 0).sum())}",
         estado="OK" if (mensual.loc[mensual.mes_completo, ["total_eeuu", "total_mexico"]] > 0).all().all() else "ERROR"),
    dict(prueba="El filtro de México nunca da más artículos que la versión sin filtro (EE.UU.)",
         resultado=f"{int((mensual.tpu_mx_eeuu > mensual.tpu_eeuu).sum())} meses con más", estado="OK" if (mensual.tpu_mx_eeuu <= mensual.tpu_eeuu).all() else "ERROR"),
])
display(checks)
if (checks.estado == "ERROR").any():
    raise RuntimeError("Hay comprobaciones en ERROR: revise la tabla antes de continuar.")
display(mensual[~mensual.mes_completo][["dias_con_datos", "dias_mes", "completitud"]])

## 7. Índices
Misma escala que el TPU: 100 = 1% de los artículos.

- `tpu_mx_gdelt_eeuu` = artículos TPU + México en medios de EE.UU. / total de artículos de medios de EE.UU. × 10,000 → **principal**
- `tpu_mx_gdelt_mexico` = lo mismo con medios mexicanos
- `tpu_mx_gdelt_global` = artículos TPU + México en todos los medios / total monitoreado por GDELT × 10,000
- `tpu_gdelt_eeuu` = sin filtro de México, medios de EE.UU. (solo para validar contra Caldara)

In [ ]:
m = mensual.copy()
ok = m.mes_completo
indices = pd.DataFrame(index=m.index)
indices["tpu_mx_gdelt_eeuu"] = (1e4 * m.tpu_mx_eeuu / m.total_eeuu).where(ok)
indices["tpu_mx_gdelt_mexico"] = (1e4 * m.tpu_mx_mexico / m.total_mexico).where(ok)
indices["tpu_mx_gdelt_global"] = (1e4 * m.tpu_mx_global / m.total_gdelt).where(ok)
indices["tpu_gdelt_eeuu"] = (1e4 * m.tpu_eeuu / m.total_eeuu).where(ok)
indices["proporcion_mexico_en_tpu_eeuu"] = (m.tpu_mx_eeuu / m.tpu_eeuu).where(ok)
display(indices.describe().T.round(3))

## 8. Validación contra el TPU de Caldara et al.
Si la traducción del método a GDELT funciona, `tpu_gdelt_eeuu` (sin filtro de México) debe parecerse al TPU original. Se comparan niveles (en logaritmos) y variaciones mensuales, y se grafican las series **estandarizadas** en un mismo eje (no se usan dos ejes).

In [ ]:
tpu_cal = pd.read_excel(RAIZ / "data/raw/incertidumbre/tpu_caldara_iacoviello.xlsx", sheet_name="TPU_MONTHLY")
tpu_cal = tpu_cal.assign(mes=pd.to_datetime(tpu_cal.DATE).dt.to_period("M").dt.to_timestamp()).set_index("mes").TPU
comp_v = pd.concat([np.log(tpu_cal.rename("tpu_caldara")), np.log(indices[["tpu_gdelt_eeuu", "tpu_mx_gdelt_eeuu", "tpu_mx_gdelt_mexico", "tpu_mx_gdelt_global"]])], axis=1, join="inner").dropna()
val = pd.DataFrame({
    "correlación con log TPU Caldara (niveles)": comp_v.corr()["tpu_caldara"],
    "correlación de variaciones mensuales": comp_v.diff().corr()["tpu_caldara"],
}).drop("tpu_caldara").round(3)
display(val)
print(f"Meses comparados: {len(comp_v)} ({comp_v.index.min():%Y-%m} a {comp_v.index.max():%Y-%m})")
if val.loc["tpu_gdelt_eeuu"].iloc[0] < 0.6:
    print("REVISAR: la versión GDELT sin filtro se parece poco al TPU de Caldara (correlación < 0.6). Revise términos y medios antes de usar el índice.")

try:
    import matplotlib.pyplot as plt
    z = (comp_v - comp_v.mean()) / comp_v.std()
    fig, ax = plt.subplots(figsize=(10, 4))
    for c, color, lw in [("tpu_caldara", "#52514e", 1.5), ("tpu_gdelt_eeuu", "#eb6834", 1.5), ("tpu_mx_gdelt_eeuu", "#2a78d6", 2.2)]:
        ax.plot(z.index, z[c], color=color, lw=lw, label={"tpu_caldara": "TPU Caldara et al.", "tpu_gdelt_eeuu": "GDELT, EE.UU., sin filtro México",
                                                           "tpu_mx_gdelt_eeuu": "GDELT, EE.UU., con filtro México"}[c])
    ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("log, estandarizado"); ax.legend(frameon=False, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False); ax.set_title("Incertidumbre de política comercial: original vs. GDELT")
    plt.tight_layout(); plt.show()
except ImportError:
    print("Instale matplotlib para ver la gráfica (pip install matplotlib).")

## 9. Meses con mayor incertidumbre comercial sobre México
Deben coincidir con episodios conocidos (mayo de 2019, noviembre de 2024, febrero-abril de 2025…).

In [ ]:
display(indices.tpu_mx_gdelt_eeuu.dropna().sort_values(ascending=False).head(12).round(1).rename("tpu_mx_gdelt_eeuu").to_frame()
        .assign(proporcion_mexico=indices.proporcion_mexico_en_tpu_eeuu.round(3)))

## 10. Guardar
- `data/interim/incertidumbre/tpu_mexico_gdelt_mensual.csv`: conteos, denominadores, completitud e índices por mes.
- `data/interim/incertidumbre/tpu_mexico_gdelt_metadatos.json`: consultas exactas, parámetros y fecha de descarga.
- La caché cruda queda en `data/raw/incertidumbre/gdelt_cache/` (súbala al repositorio: es la fuente original y permite reconstruir todo sin volver a descargar).

In [ ]:
salida = mensual.join(indices).reset_index()
salida["mes"] = salida["mes"].dt.strftime("%Y-%m")
salida.to_csv(SALIDA / "tpu_mexico_gdelt_mensual.csv", index=False)
(SALIDA / "tpu_mexico_gdelt_metadatos.json").write_text(json.dumps({
    "fuente": "GDELT DOC 2.0 API, modo timelinevolraw, resolución diaria sumada por mes",
    "url": URL, "meses": [str(MESES[0]), str(MESES[-1])], "pausa_seg": PAUSA_SEG,
    "series": {k: {"consulta": v["q"], "descripcion": v["desc"], "huella": huella(v["q"])} for k, v in SERIES.items()},
    "terminos": {"comercio": COMERCIO, "cercania": CERCANIA, "incertidumbre": INCERTIDUMBRE, "mexico": MEXICO},
    "criterio_mes_completo": "al menos 90% de los días con norm > 0 (serie global)", "clausulas_cercania": USAR_CERCANIA,
    "denominador": DECISION_DENOM, "norm_es_global": NORM_ES_GLOBAL,
    "generado": datetime.now().isoformat(timespec="seconds"), "python": platform.python_version(), "pandas": pd.__version__,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado en", SALIDA)